# Import Modules

In [1]:
import sys
from pathlib import Path

# Get the absolute path to the project root
project_root = Path.cwd().parent

# Add the project root to Python's import path
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Can now import files as if the notebook was located in the root folder
from src.data.connector_factory import ConnectorFactory
from src.models.bsm import implied_volatility
from src.models.svi import calibrate_raw_svi, svi_iv
from src.utils.time_utils import compute_time_to_expiry

import numpy as np
import plotly.graph_objects as go

# SVI Calibration
Check if SVI can be calibrated. The test fetches option data for one expiry using the Connector and fits the volatility surface using SVI

In [29]:
connector = ConnectorFactory.get_connector(provider="yfinance")
expiries = connector.get_available_expiries()[:3]

# Fetch data
spot = connector.get_spot_price()
r, q = 0.05, 0.0
expiry = expiries[0]
raw = connector.get_chain_for_expiry(expiry)
T = compute_time_to_expiry(expiry)

print(f"Expiry: {expiry}")
print(f"Time to expiry: {T:.4f} years ({T * 365:.1f} days)")
print(f"Spot Price = {spot}")

# Compute IVs
strikes, ivs = [], []
forward = spot*np.exp((r-q)*T)
for opt in raw["calls"] + raw["puts"]:
    # Filter: Only use OTM
    if opt.option_type == "call" and opt.strike > forward:
        strikes.append(opt.strike)
        ivs.append(opt.implied_vol)
    if opt.option_type == "put" and opt.strike < forward:
        strikes.append(opt.strike)
        ivs.append(opt.implied_vol)
    
# Fit SVI
k = np.log(np.array(strikes) / forward)
params = calibrate_raw_svi(k, np.array(ivs), T)
print(f"SVI params: {params}")

Expiry: 2026-07-06
Time to expiry: 0.0079 years (2.9 days)
Spot Price = 744.780029296875
SVI params: [-0.00403852  0.12107437 -0.91901963 -0.18522939  0.0852974 ]


In [30]:
# Plot
k_grid = np.linspace(k.min()-0.01, k.max()+0.01, 5000)
iv_fitted = svi_iv(k_grid, *params, T)

fig = go.Figure()
fig.add_trace(go.Scatter(x=strikes, y=ivs, mode='markers', name='Market IV'))
fig.add_trace(go.Scatter(
    x=np.exp(k_grid) * forward,
    y=iv_fitted,
    mode='lines',
    name='SVI Fit',
))
fig.update_layout(title=f"SVI Fit for {expiry}")
fig.show()